# Policy Simulation Pack 8\nSimulates baseline resolution, overrides, cap checks, expiry behavior, and invariant rejection.

In [ ]:
import os, json, requests\nBASE_URL = os.getenv('BASE_URL', 'http://localhost:3000')\nTOKEN = os.getenv('TOKEN', '')\nheaders = {'authorization': f'Bearer {TOKEN}'} if TOKEN else {}\nprint('base_url=', BASE_URL, 'token_set=', bool(TOKEN))

In [ ]:
def resolve(policy_key, client_id=None, campaign_id=None):\n    params = {'policyKey': policy_key}\n    if client_id: params['clientId'] = client_id\n    if campaign_id: params['campaignId'] = campaign_id\n    r = requests.get(f'{BASE_URL}/v1/policies/resolve', params=params, headers=headers, timeout=10)\n    return r.status_code, r.json()\n\nstatus, baseline = resolve('finance_limits')\nprint('baseline status=', status)\nprint(json.dumps(baseline, indent=2))

## Scenario templates\n1. Global margin change\n2. Client override margin\n3. Campaign CPA override with expiry\n4. Cap enforcement (11th active campaign override should fail)\n5. Invariant breach (margin < 0.25) should fail

In [ ]:
from datetime import datetime, timedelta, timezone\ndef iso(dt): return dt.astimezone(timezone.utc).isoformat().replace('+00:00','Z')\nnow = datetime.now(timezone.utc)\nexample_campaign_draft = {\n  'scopeType': 'campaign',\n  'scopeId': '11111111-1111-4111-8111-111111111111',\n  'clientId': '22222222-2222-4222-8222-222222222222',\n  'policyKey': 'performance_limits',\n  'valueJson': {\n    'maxCPA': 125,\n    'maxRouteLatencyP95': 220,\n    'maxErrorRate': 0.02,\n    'maxPodFailureRate': 0.05,\n    'minMargin': 0.3\n  },\n  'effectiveAt': iso(now - timedelta(minutes=1)),\n  'expiresAt': iso(now + timedelta(days=3)),\n  'changeReason': 'simulation',\n  'requiredRoles': ['sebastian']\n}\nprint(json.dumps(example_campaign_draft, indent=2))